# 07 · Legacy regression comparison

Diffs the tcren-reproduced outputs in `results_new/` against the 2022 mir/R baselines in
`data_legacy/` (the regression/debugging surface). Each row is one paired file with the
row-set match and the maximum numeric difference. Run the analysis notebooks first so that
`results_new/` is populated.

In [1]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, polars as pl
from tcren.paper import compare
from tcren.paper.helpers import _read_any

CONTACT_KEYS = ['pdb.id','chain.type.from','region.type.from','residue.index.from',
                'residue.index.to','residue.aa.from','residue.aa.to']

In [2]:
# Contact geometry: tcren (2022 set) vs mir, per-structure exact match on the shared structures
GEOM = ['residue.index.from','residue.index.to','residue.aa.from','residue.aa.to']
new = _read_any('results_new/contacts_2022.csv')
old = _read_any('data_legacy/contact_maps_PDB.csv.gz')
shared = sorted(set(new['pdb.id'].to_list()) & set(old['pdb.id'].to_list()))
def rs(df, pid): return set(map(tuple, df.filter(pl.col('pdb.id')==pid).select(GEOM).unique().rows()))
exact = sum(rs(new, p) == rs(old, p) for p in shared)
rows = [{'artifact': 'contact geometry (2022 vs mir)', 'n_structures': len(shared),
         'exact': exact, 'status': 'pass' if exact == len(shared) else f'{exact}/{len(shared)} exact'}]
rows[-1]

{'artifact': 'contact geometry (2022 vs mir)',
 'n_structures': 235,
 'exact': 226,
 'status': '226/235 exact'}

In [3]:
# Derived potentials vs the published 2022 matrix (correlation; not exact — different structure sets)
old_pot = _read_any('data_legacy/TCRen_potential.csv.gz').rename({'TCRen': 'value'})
for tag in ('2022', '2026'):
    new_pot = _read_any(f'results_new/TCRen_{tag}.csv')
    j = new_pot.join(old_pot, on=['residue.aa.from','residue.aa.to'], suffix='_pub')
    r = float(np.corrcoef(j['value'].to_numpy(), j['value_pub'].to_numpy())[0, 1])
    rows.append({'artifact': f'TCRen {tag} (vs published)', 'n_structures': None,
                 'exact': None, 'status': f'r={r:.3f}' + (' pass' if r > 0.8 else ' CHECK')})
rows[-1]

{'artifact': 'TCRen 2026 (vs published)',
 'n_structures': None,
 'exact': None,
 'status': 'r=0.918 pass'}

In [4]:
# Regression summary table
report = pl.DataFrame(rows)
report.write_csv('results_new/regression_report.csv')
report

artifact,n_structures,exact,status
str,i64,i64,str
"""contact geometry (2022 vs mir)""",235,226,"""226/235 exact"""
"""TCRen 2022 (vs published)""",null,null,"""r=0.846 pass"""
"""TCRen 2026 (vs published)""",null,null,"""r=0.918 pass"""
